# Institutional Research Workflow Example

**Exploration only** — reusable logic lives in `backend/loaders`, `statistical_testing`, `alpha_validation`, `feature_analysis`, and `experiments`.

Prerequisites:
```bash
cd backend
pip install -r requirements-data.txt -r requirements-research.txt
python run_data_pipeline.py --symbol "GC=F" --interval 1d --period 2y
```

In [ ]:
import sys
from pathlib import Path

BACKEND = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from loaders.dataset_loader import DatasetLoader, DatasetLoaderConfig, DatasetTier
from statistical_testing.stationarity_tests import run_stationarity_suite
from statistical_testing.distribution_tests import run_distribution_diagnostics
from alpha_validation.bootstrap_validation import run_bootstrap_validation
from feature_analysis.feature_importance import analyze_feature_importance
from experiments.experiment_tracker import ExperimentTracker

SYMBOL = "GC=F"
INTERVAL = "1d"

In [ ]:
loader = DatasetLoader(DatasetLoaderConfig(tier=DatasetTier.FEATURES))
df = loader.load(SYMBOL, INTERVAL)
returns = df["log_return"].dropna()
print(f"Loaded {len(df)} bars from {loader.resolve_path(SYMBOL, INTERVAL)}")

In [ ]:
stationarity = run_stationarity_suite(returns)
distribution = run_distribution_diagnostics(returns)

print("Stationarity:", stationarity.summary)
print("Distribution:", distribution.summary)
if stationarity.warnings:
    print("Warnings:", stationarity.warnings)

In [ ]:
signal = (df["z_score"].shift(1) > 0).astype(float)
strategy_returns = (signal * df["pct_return"]).dropna()

bootstrap = run_bootstrap_validation(strategy_returns)
print(bootstrap.summary)
print(bootstrap.sharpe.interpretation)

In [ ]:
feature_cols = ["z_score", "rolling_volatility", "atr"]
X = df[feature_cols].dropna()
y = df.loc[X.index, "pct_return"].shift(-1)

importance = analyze_feature_importance(X.iloc[:-1], y.iloc[:-1], compute_rolling=False)
print(importance.summary)
print(importance.disclaimer)

In [ ]:
tracker = ExperimentTracker()
record = tracker.create_experiment(
    dataset_version=f"{SYMBOL}_{INTERVAL}_features",
    features_used=feature_cols,
    parameters={"signal": "z_score_lag1", "tier": "features"},
    metrics={
        "bootstrap_sharpe": bootstrap.sharpe.observed,
        "sharpe_robust": bootstrap.sharpe.is_robust,
    },
    notes="Example workflow — not a production strategy",
    tags=["example", "phase3"],
)
print("Experiment ID:", record.experiment_id)